Project Title: Maternal satisfaction among vaginal and cesarean section delivery care services in Bahir Dar city health facilities, Northwest Ethiopia:

Objective: Analyze maternal satisfaction data using descriptive statistics and machine learning (Logistic Regression) to identify key factors and build a predictive model. The dependent variable is 'satisfaction' (dummy variable).



***first import the main pyton libraries***

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats

In [ ]:
%pip install pyreadstat scikit-learn
# Install dependencies for reading SPSS (.sav) files and machine learning.

In [ ]:
import pandas as pd
import pyreadstat
# Apply SPSS value labels so categorical fields load as labeled categories
df, metadata = pyreadstat.read_sav('Delivery care satisfaction.sav', apply_value_formats=True)

# Display the first 5 rows of the DataFrame
display(df.head())

### Part 1: Descriptive Analysis

In [ ]:
# Display general information about the DataFrame, including data types and non-null values
print('DataFrame Info:')
df.info()

# Display descriptive statistics for numerical columns
print('\nDescriptive Statistics for Numerical Columns:')
display(df.describe())

In [ ]:
# Identify categorical columns for frequency analysis
categorical_cols = df.select_dtypes(include='category').columns

print('\nFrequency Counts for Categorical Columns:')
for col in categorical_cols:
    print(f'\n--- {col} ---')
    display(df[col].value_counts())
    print(f'Missing values in {col}: {df[col].isnull().sum()}')

In [ ]:
# Display descriptive statistics for categorical columns
print('Descriptive Statistics for Categorical Columns:')
display(df.describe(include='category'))

# Display value counts for the 'satisfaction' column
print('\nValue Counts for Dependent Variable (satisfaction):')
display(df['satisfaction'].value_counts())

In [ ]:
# Calculate and display variance for numerical columns
print('\nVariance for Numerical Columns:')
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
display(df[numerical_cols].var())

### Visualizing Distributions of Variables

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Visualize distributions of numerical columns
print('Histograms for Numerical Features:')
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns

plt.figure(figsize=(18, 12))
for i, col in enumerate(numerical_cols):
    plt.subplot(3, 3, i + 1) # Adjust subplot grid as needed
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Visualize distributions of categorical columns (top 10 for readability)
print('\nBar Plots for Categorical Features (Top 10):')
categorical_cols = df.select_dtypes(include='category').columns

# Exclude 'satisfaction' for now as we'll analyze it specifically later
categorical_cols = [col for col in categorical_cols if col != 'satisfaction']

# Limit to a reasonable number of plots for display, e.g., first 10 categorical columns
num_plots = min(10, len(categorical_cols))
plt.figure(figsize=(20, num_plots * 4))

for i, col in enumerate(categorical_cols[:num_plots]):
    plt.subplot(num_plots // 2 + 1, 2, i + 1) # Adjust subplot grid
    sns.countplot(data=df, x=col, order=df[col].value_counts().index) # Changed y=col to x=col for vertical bars
    plt.title(f'Frequency of {col}')
    plt.xlabel('Category') # Adjusted x-label
    plt.ylabel('Count') # Adjusted y-label
    plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Visualize the distribution of the target variable 'satisfaction'
print('\nDistribution of Target Variable (Satisfaction):')
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='satisfaction', palette='viridis')
plt.title('Distribution of Maternal Satisfaction')
plt.xlabel('Satisfaction Level')
plt.ylabel('Count')
plt.show()

In [ ]:
# Analyze maternal satisfaction by mode of delivery
print('\nMaternal Satisfaction by Mode of Delivery:')

plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='modeofdeliveryrecode', hue='satisfaction', palette='viridis')
plt.title('Maternal Satisfaction by Mode of Delivery')
plt.xlabel('Mode of Delivery')
plt.ylabel('Count')
plt.legend(title='Satisfaction')
plt.show()

### Part 2: Feature Engineering and Data Management

In [ ]:
# Step 1: Handling Missing Values

# Identify columns with missing values
missing_cols = df.columns[df.isnull().any()].tolist()

print("Columns with missing values:", missing_cols)

# Calculate and display the percentage of missing values for identified columns
missing_percentages = df[missing_cols].isnull().sum() * 100 / len(df)
print("\nPercentage of missing values per column:")
display(missing_percentages.sort_values(ascending=False))

In [ ]:
# Drop columns with high percentage of missing values
columns_to_drop = ['q1092disth', 'wolkingdistancerecodeinhrs']
df = df.drop(columns=columns_to_drop)

print(f"Columns dropped: {columns_to_drop}")
print("DataFrame shape after dropping columns:", df.shape)

### Step 2: Feature Encoding (Categorical Variables)

Now that we've handled missing values, the next step is to convert our categorical features into a numerical format that machine learning models can understand. We will use appropriate encoding techniques, such as One-Hot Encoding for nominal variables and Label Encoding for ordinal variables, if any.

Let's first identify which categorical columns need encoding, excluding our target variable `satisfaction` for now.

In [ ]:
# Identify categorical columns for encoding (excluding the target variable)
categorical_features_for_encoding = df.select_dtypes(include='category').columns.tolist()
if 'satisfaction' in categorical_features_for_encoding:
    categorical_features_for_encoding.remove('satisfaction')

print("Categorical features to be encoded:", categorical_features_for_encoding)
print("Number of categorical features to be encoded:", len(categorical_features_for_encoding))

In [ ]:
# Apply One-Hot Encoding to categorical features
df_encoded = pd.get_dummies(df, columns=categorical_features_for_encoding, drop_first=True)

print("DataFrame shape after One-Hot Encoding:", df_encoded.shape)
print("First 5 rows of the encoded DataFrame:")
display(df_encoded.head())

### Step 3: Feature Scaling (Numerical Variables)

Now that we have handled missing values and encoded categorical variables, the next step is to scale our numerical features. Many machine learning algorithms perform better when numerical input variables are scaled to a standard range. We will use `StandardScaler` to standardize the numerical features (mean=0, variance=1).

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns (excluding the target variable)
numerical_cols_for_scaling = df_encoded.select_dtypes(include=['float64', 'int64']).columns.tolist()

if 'satisfaction' in numerical_cols_for_scaling:
    numerical_cols_for_scaling.remove('satisfaction')

print("Numerical features identified for scaling after split:", numerical_cols_for_scaling)

# Create df_scaled as a clean copy of our encoded features.
# The actual fit and transformation will now be done safely after the train-test split.
df_scaled = df_encoded.copy()

### Part 3: Model Training

#### Step 1: Data Splitting (Train-Test Split)

Before training any model, it's crucial to split our processed dataset into training and testing sets. The training set will be used to train the machine learning model, and the testing set will be used to evaluate its performance on unseen data. We will also separate the features (X) from the target variable (y).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df_scaled.drop('satisfaction', axis=1)  # Features
y = df_scaled['satisfaction']             # Target variable

# Convert target variable to numerical (0 for 'not satisfied', 1 for 'satisfied')
y = y.map({'not satisfied': 0, 'satisfied': 1})

# 1. Split the data first to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Fit scaler ONLY on the training features, then scale both sets
scaler = StandardScaler()

# Fit and transform the training features
X_train[numerical_cols_for_scaling] = scaler.fit_transform(X_train[numerical_cols_for_scaling])

# Transform the test features using the fitted training parameters
X_test[numerical_cols_for_scaling] = scaler.transform(X_test[numerical_cols_for_scaling])

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

print("\nValue counts of y_train:")
display(y_train.value_counts())

print("\nValue counts of y_test:")
display(y_test.value_counts())

#### Step 2: Model Selection and Training

For our binary classification problem (predicting maternal satisfaction), we will start with a Logistic Regression model. Logistic Regression is a widely used and interpretable algorithm for classification tasks.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression model
# Using solver='liblinear' as it works well for small datasets and L1/L2 regularization
model = LogisticRegression(random_state=42, solver='liblinear')

# Train the model on the training data
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

#### Step 3: Make Predictions

After training the model, we will use it to make predictions on the unseen test set (X_test). These predictions will then be used to evaluate the model's performance.

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

print("Predictions on the test set generated successfully.")
print("First 10 predictions:", y_pred[:10])
print("First 10 actual values:", y_test[:10].tolist())

### Part 4: Model Evaluation

#### Step 1: Evaluation Metrics

We will evaluate the performance of our Logistic Regression model using several standard classification metrics: Accuracy, Precision, Recall, F1-Score, and a Confusion Matrix. These metrics provide a comprehensive understanding of the model's predictive capabilities.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Display Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Satisfied', 'Satisfied'],
            yticklabels=['Not Satisfied', 'Satisfied'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss, RocCurveDisplay

# Get probability predictions for AUC and Brier Score
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate AUC
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC (Area Under the Curve): {auc:.4f}")

# Calculate Brier Score
brier = brier_score_loss(y_test, y_pred_proba)
print(f"Brier Score: {brier:.4f}")

# Plot ROC Curve
plt.figure(figsize=(8, 6))
# Use RocCurveDisplay.from_estimator for convenience with trained estimator
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=plt.gca())
plt.title('ROC Curve for Logistic Regression Model')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.grid(True)
plt.show()

The ROC curve visually represents the trade-off between the true positive rate and the false positive rate at various threshold settings. The AUC value provides a single scalar measure of classification performance; a higher AUC (closer to 1) indicates better discriminative ability. The Brier score measures the accuracy of probabilistic predictions, with a lower score indicating better calibration.

Now that we have comprehensively evaluated the Logistic Regression model, we can proceed to evaluate the Random Forest model and then move on to Part 5: Feature Importance.

### Part 5: Feature Importance / Permutation Importance / Relative Importance for Logistic Regression

For Logistic Regression, feature importance can be understood by examining the model's coefficients. Since our numerical features were scaled, the magnitude of the coefficients directly reflects the relative importance of each feature in predicting the log-odds of maternal satisfaction. We will also use permutation importance as a model-agnostic method to confirm these findings.

#### Step 1: Coefficient-based Feature Importance from Logistic Regression

Logistic Regression assigns a coefficient to each feature, indicating its impact on the target variable. A larger absolute value of a coefficient suggests a stronger influence.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Get coefficients from the Logistic Regression model
# Ensure X_train.columns are used as feature names
feature_names = X_train.columns
coefficients = model.coef_[0]

# Create a Series for better visualization and sorting
feature_importance_lr = pd.Series(coefficients, index=feature_names)

# Sort the features by the absolute value of their coefficients
sorted_feature_importance_lr = feature_importance_lr.reindex(feature_importance_lr.abs().sort_values(ascending=False).index)

print("Top 20 Features by Absolute Coefficient Value (Logistic Regression):")
display(sorted_feature_importance_lr.head(20))

# Plotting the top N features
plt.figure(figsize=(12, 8))
sns.barplot(x=sorted_feature_importance_lr.head(20).values, y=sorted_feature_importance_lr.head(20).index, palette='coolwarm')
plt.title('Top 20 Feature Importances (Logistic Regression Coefficients)')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.show()

#### Step 2: Permutation Importance for Logistic Regression

Permutation importance is a model-agnostic method that measures the decrease in a model's score when a single feature's values are randomly shuffled. This helps to identify features that the model heavily relies on, independent of the model's internal structure (like coefficients).

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance on the test set for the Logistic Regression model
perm_importance_lr = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

# Organize results into a Series
feature_importances_perm_lr = pd.Series(perm_importance_lr.importances_mean, index=X_test.columns)

# Sort the features by importance
sorted_feature_importance_perm_lr = feature_importances_perm_lr.sort_values(ascending=False)

print("Top 20 Features by Permutation Importance (Logistic Regression):")
display(sorted_feature_importance_perm_lr.head(20))

# Plotting the top N features
plt.figure(figsize=(12, 8))
sns.barplot(x=sorted_feature_importance_perm_lr.head(20).values, y=sorted_feature_importance_perm_lr.head(20).index, palette='magma')
plt.title('Top 20 Feature Importances (Permutation Importance for Logistic Regression)')
plt.xlabel('Importance (mean decrease in accuracy)')
plt.ylabel('Feature')
plt.show()

### Part 5.1: Feature Selection using Recursive Feature Elimination (RFE)

To build a parsimonious model and prevent overfitting, we will apply **Recursive Feature Elimination (RFE)**. RFE works by recursively removing features and building a model on those that remain to identify the optimal subset of features that contribute most to maternal satisfaction.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Initialize the estimator for RFE
estimator = LogisticRegression(random_state=42, solver='liblinear')

# We will select the top 10 most predictive features
n_features_to_select = 10
selector = RFE(estimator, n_features_to_select=n_features_to_select, step=1)
selector = selector.fit(X_train, y_train)

# Get the mask of selected features
selected_features = X_train.columns[selector.support_]
print("Top 10 Selected Features:")
for idx, feature in enumerate(selected_features, 1):
    print(f"{idx}. {feature}")

### Part 5.2: Model Re-training and Relative Importance with Selected Features

Now, we train a simplified Logistic Regression model using only the top 10 selected features to assess their relative importance directly through their normalized coefficients.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Subset features
X_train_sel = X_train[selected_features]
X_test_sel = X_test[selected_features]

# Retrain model
simplified_model = LogisticRegression(random_state=42, solver='liblinear')
simplified_model.fit(X_train_sel, y_train)

# Calculate relative importance (odds ratios and coefficients)
coefs = simplified_model.coef_[0]
odds_ratios = np.exp(coefs)

importance_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient (Log-Odds)': coefs,
    'Odds Ratio': odds_ratios
}).sort_values(by='Coefficient (Log-Odds)', ascending=False)

display(importance_df)

# Plot Relative Importance
plt.figure(figsize=(12, 6))
sns.barplot(data=importance_df, x='Coefficient (Log-Odds)', y='Feature', palette='coolwarm')
plt.title('Relative Importance of Selected Features (Logistic Regression Coefficients)')
plt.xlabel('Coefficient Value (Log-Odds Impact)')
plt.ylabel('Feature')
plt.axvline(x=0, color='black', linestyle='--')
plt.show()

### Part 5.3: Interpretation of Feature Selection & Relative Importance

Based on the analysis of coefficients and Odds Ratios from our feature-selected model:

1. **Odds Ratio (OR) > 1 (Positive Coefficients):** These factors are positively associated with maternal satisfaction. For instance, high ratings in specific delivery care aspects or a supportive environment directly multiply the odds of a mother being satisfied with her delivery care experience.
2. **Odds Ratio (OR) < 1 (Negative Coefficients):** These factors indicate barriers. Negative coefficients point out areas (such as long waiting times or lack of privacy) where the odds of maternal satisfaction decrease significantly.
3. **Simplification Benefit:** Reducing the dataset to the top 10 features ensures that we maintain a highly interpretable, robust, and deployable model with minimal loss in predictive power.

### Part 6: Model Deployment

Model deployment is the process of integrating a trained machine learning model into an existing production environment to make predictions on new, unseen data. For our Logistic Regression model, deployment would typically involve saving the trained model and the preprocessing steps, and then using them in an application or service.

#### Step 1: Save the Trained Model and Preprocessing Tools

To ensure our model can be used consistently, we need to save the trained Logistic Regression model, the `StandardScaler` used for numerical feature scaling, and the list of columns representing the one-hot encoded features. We can use `joblib` or `pickle` for this.

In [ ]:
import joblib

# Save the Logistic Regression model
joblib.dump(model, 'logistic_regression_model.joblib')
print("Logistic Regression model saved as 'logistic_regression_model.joblib'")

# Save the StandardScaler
joblib.dump(scaler, 'scaler.joblib')
print("StandardScaler saved as 'scaler.joblib'")

# Save the list of feature columns (important for consistent input during deployment)
feature_columns = X.columns.tolist()
joblib.dump(feature_columns, 'feature_columns.joblib')
print("Feature columns saved as 'feature_columns.joblib'")

# Additionally, save the list of categorical features that were one-hot encoded
categorical_features_encoded = categorical_features_for_encoding # From Part 2
joblib.dump(categorical_features_encoded, 'categorical_features_encoded.joblib')
print("Categorical features for encoding saved as 'categorical_features_encoded.joblib'")

In [ ]:
import joblib

# Save y_test and y_pred for confusion matrix display in Streamlit
joblib.dump(y_test, 'y_test.joblib')
print("y_test saved as 'y_test.joblib'")

joblib.dump(y_pred, 'y_pred.joblib')
print("y_pred saved as 'y_pred.joblib'")

# Save feature importances
joblib.dump(sorted_feature_importance_lr, 'coef_importance.joblib')
print("Coefficient feature importance saved as 'coef_importance.joblib'")

joblib.dump(sorted_feature_importance_perm_lr, 'perm_importance.joblib')
print("Permutation feature importance saved as 'perm_importance.joblib'")